# Creating Train and Test Sets with Implicit Feedback
- Dataset: MovieLens32m
- Split Method: leave-one-out + 100 negatives

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

np.random.seed(42)

In [2]:
def set_dtypes(dataset: pd.DataFrame) -> pd.DataFrame:
    dataset['userId'] = dataset['userId'].astype(np.uint32)
    dataset['rating'] = dataset['rating'].astype(np.float16)
    dataset['movieId'] = dataset['movieId'].astype(np.uint32)
    dataset['timestamp'] = dataset['timestamp'].astype(np.uint32)

    return dataset

In [3]:
ratings = pd.read_csv('../data/ml-32m/ratings.csv')
ratings = set_dtypes(ratings)

In [4]:
user_pool = set(ratings['userId'].unique())
item_pool = set(ratings['movieId'].unique())

### Remap IDs to be contiguous

In [5]:
user_id_map = {uid: idx for idx, uid in enumerate(ratings['userId'].unique())}
item_id_map = {mid: idx for idx, mid in enumerate(ratings['movieId'].unique())}

ratings['userId'] = ratings['userId'].map(user_id_map).astype(np.int64)
ratings['movieId'] = ratings['movieId'].map(item_id_map).astype(np.int64)

### Pop last rated item

In [6]:
def pop_last_user_review(ratings_df):
    # Sort by timestamp to ensure the last review is at the end
    ratings_df = ratings_df.sort_values(by='timestamp')
    
    # Get the last review for each user
    last_reviews = ratings_df.groupby('userId').tail(1)
    
    # Remove the last reviews from the original DataFrame
    ratings_without_last = ratings_df[~ratings_df.index.isin(last_reviews.index)]
    
    return ratings_without_last, last_reviews

In [7]:
ratings_without_last, last_ratings = pop_last_user_review(ratings)

### Get Negatives

In [8]:
def sample_negatives(ratings, item_pool, num_neg=99):
    """
    For each user, sample `num_neg` negative items (items the user hasn't interacted with)
    using rejection sampling to avoid materializing the full negative set per user.
    """
    item_list = np.array(list(item_pool))
    num_items = len(item_list)

    # Build a set of interacted items per user (compact representation)
    user_positives = ratings.groupby('userId')['movieId'].apply(set)

    neg_samples = {}
    for user_id, pos_set in user_positives.items():
        sampled = set()
        while len(sampled) < num_neg:
            candidates = item_list[np.random.randint(0, num_items, size=num_neg - len(sampled))]
            for c in candidates:
                if c not in pos_set:
                    sampled.add(c)
                    if len(sampled) == num_neg:
                        break
        neg_samples[user_id] = list(sampled)

    return neg_samples

In [9]:
negatives = sample_negatives(ratings, item_pool, 100)

### Create a Pandas DataFrame of Negatives

In [10]:
# Create a pandas dataframe with userId, negativeSamples
negatives_df = pd.DataFrame([(user_id, negs) for user_id, negs in negatives.items()], columns=['userId', 'negativeSamples'])

In [11]:
negatives_df.head()

,userId,negativeSamples
0,0,"[279560, 186377, 282121, 201232, 77332, 176149..."
1,1,"[199179, 249356, 132628, 242716, 123421, 16900..."
2,2,"[66051, 156675, 259589, 141325, 156175, 2578, ..."
3,3,"[3072, 170511, 263699, 6679, 207896, 123423, 1..."
4,4,"[139779, 180227, 279056, 198675, 4632, 180761,..."


### Concatenate with leave-one-out

In [12]:
negatives_and_leave_one_out_df = pd.merge(last_ratings[['userId', 'movieId']], negatives_df, on='userId')

In [13]:
negatives_and_leave_one_out_df.head()

,userId,movieId,negativeSamples
0,124330,34572,"[1542, 176655, 251410, 239132, 113186, 144418,..."
1,39587,1540,"[7168, 281604, 207367, 166409, 288779, 182799,..."
2,50822,8548,"[77831, 6679, 172063, 33316, 150054, 67624, 12..."
3,35010,166,"[54274, 101897, 264201, 106506, 144910, 67087,..."
4,58831,428,"[262657, 169998, 8207, 65556, 209941, 158230, ..."


### Binarize training set

In [14]:
ratings_without_last['rating'] = ratings_without_last['rating'].apply(lambda x: 1.0 if x > 0 else 0.0) # Same as the authors did, so all ratings are interactions (1.0) and the rest are non-interactions (0.0)
# Also drop the timestamp column as it's not needed for training
ratings_without_last = ratings_without_last.drop(columns=['timestamp'])


In [15]:
ratings_without_last.head()

,userId,movieId,rating
3994589,25061,4253,1.0
4958006,30916,377,1.0
4957957,30916,345,1.0
4957963,30916,322,1.0
6989318,43719,1013,1.0


### Export as `.csv`

In [16]:
import os
os.makedirs('../data/implicit', exist_ok=True)
negatives_and_leave_one_out_df.to_csv('../data/implicit/test.csv', index=False, sep=';')

In [17]:
ratings_without_last.to_csv('../data/implicit/train.csv', index=False)

# Create smaller subsets of data

In [4]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/implicit', exist_ok=True)

train = pd.read_csv('../data/implicit/train.csv', sep=',')
test = pd.read_csv('../data/implicit/test.csv', sep=';')

# Sample ~10% of users to keep train/test alignment
np.random.seed(42)
all_users = test['userId'].unique()
subset_users = np.random.choice(all_users, size=int(len(all_users) * 0.1), replace=False)

train_subset = train[train['userId'].isin(subset_users)]
test_subset = test[test['userId'].isin(subset_users)]

train_subset.to_csv('../data/implicit/train_subset.csv', index=False)
test_subset.to_csv('../data/implicit/test_subset.csv', index=False, sep=';')

print(f'Train subset: {len(train_subset)} rows, {train_subset.userId.nunique()} users')
print(f'Test subset: {len(test_subset)} rows, {test_subset.userId.nunique()} users')

Train subset: 3262062 rows, 20094 users
Test subset: 20094 rows, 20094 users
